# 03 — Inspecting Model Behavior

You have a bug. The model gave you the wrong answer. Where did it go wrong?

Was the prompt malformed? Did you hit the context limit and truncate silently? Did retrieval return the wrong documents? Did the memory system surface irrelevant context?

`llm_inspector` makes these questions answerable. It's not about wrapping LLM calls — it's about **inspecting context assembly**. When you build a prompt from memory, retrieval, or conversation history, `llm_inspector` shows you what made it into the final context, what was dropped, and why.

## Exercise: Discover the API

Before we dive in, practice a skill you'll use constantly when working with unfamiliar libraries: API discovery. Instead of searching docs or asking an LLM, interrogate the module directly.

In [1]:
import llm_inspector

# What's exported at the top level?
exports = [name for name in dir(llm_inspector) if not name.startswith('_')]
print(f"Found {len(exports)} exports\n")

# Group by type
types = {}
for name in exports:
    obj = getattr(llm_inspector, name)
    type_name = type(obj).__name__
    types.setdefault(type_name, []).append(name)

for type_name in sorted(types):
    print(f"{type_name}:")
    for name in sorted(types[type_name]):
        print(f"  - {name}")
    print()

Found 50 exports

_AnyMeta:
  - Any

_ProtocolMeta:
  - BaselineAugmenter
  - ContextAugmenter
  - EngramAugmenter
  - EngramLiteAugmenter

function:
  - build_bundle
  - bundle_to_dict
  - bundle_to_json
  - describe_inspector
  - diff_to_dict
  - diff_to_json
  - diff_traces
  - import_module
  - make_engram
  - make_engram_lite
  - render_comparison
  - render_diff
  - report_to_dict
  - report_to_json
  - report_to_operation_result
  - trace_to_interop_events
  - trace_to_memory_records
  - trace_to_operation_result

module:
  - adapters
  - augmenters
  - core
  - export
  - inspectors
  - interop
  - protocols
  - renderers

type:
  - AdapterRegistry
  - AdapterSpec
  - AugmentRequest
  - ChromaDBRAGAdapter
  - CompareBundle
  - ComparisonReport
  - ContextInspector
  - ContextResult
  - DiffReport
  - EngramRAGAdapter
  - EvidenceItem
  - NamedTrace
  - RAGInspector
  - RunMetrics
  - Section
  - TokenAccounting
  - Trace
  - TraceEvent
  - Turn



**What to look for:**
- Data models (classes with `Trace`, `Evidence`, `Context` in the name)
- Functions for comparison (`diff`, `compare`, `render`)
- Augmenters (things that build context)

Take a minute to scan the output. Which names suggest 'this shows me what happened'?

## What you should have found

The key pieces:

**Core types:**
- `Trace` — a record of one turn: the user message, assembled context, and metrics
- `ContextResult` — what context was assembled (sections, evidence, token accounting)
- `EvidenceItem` — a fact or source the context used
- `Section` — a chunk of context (e.g., from memory, from retrieval, from system prompt)

**Inspection tools:**
- `diff_traces()` — compare two traces, find what changed
- `render_diff()` — pretty-print the diff
- `render_comparison()` — show two traces side-by-side

**Context builders (augmenters):**
- `BaselineAugmenter` — minimal context (just the user message)
- `EngramAugmenter` / `EngramLiteAugmenter` — add memory (notebook 04)
- `RAGInspector` — add retrieved documents (notebook 05)

The workflow: an augmenter builds context and produces a `Trace`. You compare traces to see what differed.

## Anatomy of a Trace

A `Trace` has three parts:
1. **turn** — the user's message (role, text, session ID, timestamp)
2. **context** — what was assembled into the prompt
3. **metrics** — model, latency, token counts, errors (if you ran the model)

Let's build a minimal trace by hand to see the structure:

In [2]:
from llm_inspector import Trace, Turn, ContextResult, Section, TokenAccounting, RunMetrics

# Build a trace manually
trace = Trace(
    turn=Turn(
        role="user",
        text="What's the difference between `is` and `==` in Python?",
        session_id="demo_session",
    ),
    context=ContextResult(
        sections=[
            Section(
                title="System Prompt",
                text="You are a Python expert. Be concise.",
                origin="system",
                tokens=12,
            ),
            Section(
                title="User Message",
                text="What's the difference between `is` and `==` in Python?",
                origin="user",
                tokens=18,
            ),
        ],
        token_accounting=TokenAccounting(
            target_tokens=1000,
            total_tokens=30,
            truncated=False,
        ),
    ),
    metrics=RunMetrics(
        model="qwen3.5:9b",
        latency_ms=450.0,
        prompt_tokens=30,
        output_tokens=85,
    ),
)

# Inspect it
print(f"Turn: {trace.turn.text[:50]}...")
print(f"Sections: {len(trace.context.sections)}")
print(f"Total tokens: {trace.context.token_accounting.total_tokens}")
print(f"Model: {trace.metrics.model}")
print(f"Latency: {trace.metrics.latency_ms}ms")

Turn: What's the difference between `is` and `==` in Pyt...
Sections: 2
Total tokens: 30
Model: qwen3.5:9b
Latency: 450.0ms


This is what augmenters produce. When you use `BaselineAugmenter` or `EngramAugmenter` to build context, you get back a trace like this showing exactly what went into the prompt.

## Comparing traces

The real power: side-by-side comparison. Why did the same user message produce different context?

Common causes:
- Memory changed (new facts learned, old facts forgotten)
- Retrieval returned different documents
- Token budget forced truncation
- System prompt changed

Let's create two traces and diff them:

In [3]:
# Trace A: baseline (no memory)
trace_a = Trace(
    turn=Turn(role="user", text="What did we decide about the API design?", session_id="s1"),
    context=ContextResult(
        sections=[
            Section(title="User", text="What did we decide about the API design?", origin="user", tokens=12),
        ],
        token_accounting=TokenAccounting(total_tokens=12),
    ),
)

# Trace B: with memory (found a relevant prior conversation)
trace_b = Trace(
    turn=Turn(role="user", text="What did we decide about the API design?", session_id="s1"),
    context=ContextResult(
        sections=[
            Section(
                title="Memory (from 2 hours ago)",
                text="User: Should we use REST or GraphQL?\nAssistant: We decided on REST for simplicity.",
                origin="memory",
                tokens=28,
            ),
            Section(title="User", text="What did we decide about the API design?", origin="user", tokens=12),
        ],
        token_accounting=TokenAccounting(total_tokens=40),
    ),
)

# Compare them
from llm_inspector import diff_traces, render_diff

diff = diff_traces(trace_a, trace_b)
print(render_diff(diff))

Diff: A -> B

Flags:
- total_tokens: 12 -> 40

Sections:
+ [memory] Memory (from 2 hours ago) (tokens=28)



The diff shows exactly what changed: trace B has an extra section from memory that trace A lacked. Token count increased from 12 to 40. The diff makes the difference visible.

## Real debugging scenario: context overflow

You're building the code-review assistant. In dev with short files, it works. In prod with a large codebase, responses become generic and useless. What happened?

Hypothesis: token budget forced truncation, cutting out the relevant code context.

In [4]:
# Simulate dev trace (small file, all context fits)
dev_trace = Trace(
    turn=Turn(role="user", text="Review this function for bugs", session_id="dev"),
    context=ContextResult(
        sections=[
            Section(title="File Context", text="def process(data):\n    return data.upper()",
                    origin="retrieval", tokens=15),
            Section(title="User", text="Review this function for bugs", origin="user", tokens=8),
        ],
        token_accounting=TokenAccounting(
            target_tokens=1000,
            total_tokens=23,
            truncated=False,
        ),
    ),
)

# Simulate prod trace (huge file, context truncated)
prod_trace = Trace(
    turn=Turn(role="user", text="Review this function for bugs", session_id="prod"),
    context=ContextResult(
        sections=[
            Section(title="User", text="Review this function for bugs", origin="user", tokens=8),
        ],
        token_accounting=TokenAccounting(
            target_tokens=1000,
            total_tokens=8,
            truncated=True,
            notes=["Dropped 'File Context' section (15000 tokens) to fit budget"],
        ),
    ),
)

# Diff them
diff = diff_traces(dev_trace, prod_trace)
print("=== Dev vs Prod ===")
print(f"Dev sections: {len(dev_trace.context.sections)}")
print(f"Prod sections: {len(prod_trace.context.sections)}")
print(f"Prod truncated: {prod_trace.context.token_accounting.truncated}")
print(f"Truncation note: {prod_trace.context.token_accounting.notes[0]}")
print("\nThe bug: retrieval found the file but token budget cut it.")

=== Dev vs Prod ===
Dev sections: 2
Prod sections: 1
Prod truncated: True
Truncation note: Dropped 'File Context' section (15000 tokens) to fit budget

The bug: retrieval found the file but token budget cut it.


Without traces, you'd just see 'prod responses are bad' and spend hours debugging. With traces, you see immediately: `truncated=True` and the note tells you exactly what was dropped and why.

## When to use llm_inspector

Use it when:
- **Building context assembly logic.** Memory, retrieval, prompt templates — you need to see what actually made it in.
- **Debugging production failures.** It worked yesterday, broke today — diff the traces.
- **Optimizing token budgets.** Where are the tokens going? Which sections are worth keeping?
- **Comparing retrieval strategies.** BM25 vs dense vs hybrid — which returned better context?

Skip it for:
- Simple single-turn queries with no context assembly
- Production hot paths where trace overhead matters
- Cases where logging the final prompt is enough

## Augmenters: the trace source

In practice, you don't build traces by hand. You use an **augmenter** — something that takes a user message and builds context around it.

The simplest: `BaselineAugmenter` (just passes the user message through).

In [6]:
from llm_inspector import BaselineAugmenter, AugmentRequest, Turn

augmenter = BaselineAugmenter()
request = AugmentRequest(
    turn=Turn(role="user", text="Hello, world", session_id="test"),
)

trace = augmenter.augment(request)
print(f"Sections: {len(trace.context.sections)}")
print(f"Text: {trace.context.sections[0].text}")             # System prompt 
print(f"Origin: {trace.context.sections[0].origin}")
print(f"Text: {trace.context.sections[1].text}")             # User prompt
print(f"Origin: {trace.context.sections[1].origin}")

Sections: 2
Text: You are a helpful assistant.
Origin: system
Text: Hello, world
Origin: user


`BaselineAugmenter` is the control group. Notebooks 04-05 introduce augmenters that add memory and retrieval — that's where traces become essential for understanding what context was assembled and why.

## Artifact: trace comparison tool

Build a helper that compares two traces and surfaces the key differences. This is what you'd use in a debugging session.

In [7]:
def compare_context(trace_a: Trace, trace_b: Trace) -> dict:
    """
    Compare two traces and return key differences.
    
    Returns:
        dict with 'section_count_changed', 'tokens_changed', 'truncation_changed',
        'sections_added', 'sections_removed'
    """
    sections_a = {s.title for s in trace_a.context.sections}
    sections_b = {s.title for s in trace_b.context.sections}
    
    return {
        "section_count_changed": len(trace_a.context.sections) != len(trace_b.context.sections),
        "tokens_changed": (
            trace_a.context.token_accounting.total_tokens !=
            trace_b.context.token_accounting.total_tokens
        ),
        "truncation_changed": (
            trace_a.context.token_accounting.truncated !=
            trace_b.context.token_accounting.truncated
        ),
        "sections_added": list(sections_b - sections_a),
        "sections_removed": list(sections_a - sections_b),
    }

# Test it
diff_summary = compare_context(dev_trace, prod_trace)
print("Dev vs Prod differences:")
for key, value in diff_summary.items():
    print(f"  {key}: {value}")

Dev vs Prod differences:
  section_count_changed: True
  tokens_changed: True
  truncation_changed: True
  sections_added: []
  sections_removed: ['File Context']


## What's next

You now understand:
- How to discover an unfamiliar library's API (`dir()`, type inspection)
- What `llm_inspector` is for (context assembly inspection, not LLM call wrapping)
- The trace model (turn + context + metrics)
- How to compare traces to debug context issues

Notebook 04 introduces **memory with `engram_lite`**. Memory is context assembly — you have a long conversation, the context window is finite, so you select which past turns to include. `llm_inspector` traces show you exactly what the memory system chose and why.

Notebook 05 does the same for **RAG** — retrieval selects documents, `llm_inspector` shows which ones made it into the context and which were dropped.

## Exercises

1. **Build a trace from a real conversation.** Take a 5-turn conversation, pick one turn, manually construct a `Trace` showing what context you'd assemble. Include sections for: system prompt, 2 relevant prior turns, current user message.

2. **Simulate truncation.** Create two traces where the second one has `truncated=True` and a missing section. Use `diff_traces()` and `render_diff()` to visualize the difference.

3. **Explore augmenters.** Try `BaselineAugmenter`, `EngramLiteAugmenter` (if you have `engram_lite` set up), and compare their traces for the same input. What sections does each one produce?

4. **Extend compare_context().** Add checks for: model changed, latency changed, evidence items changed. Make it return a human-readable summary string.

---
**Next:** [04 — Memory with engram_lite](04_memory_with_engram_lite.ipynb) tackles conversation-too-long problems using layered memory strategies, with `llm_inspector` traces showing exactly what the memory system surfaced.